# Phase 10.1 — Final Launcher Audit

Verifies the finalized training launcher without starting training or modifying the dataset, model architecture, augmentation, loss, class weights, or approved hyperparameters.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from weapon_threat_detection.artifacts import utc_timestamp, write_json
from weapon_threat_detection.training_launcher import smoke_test

result = smoke_test(ROOT)
required = [
    "configuration_loading",
    "dataset_loading",
    "model_construction",
    "pretrained_weight_loading",
    "cbam_initialization",
    "focal_loss_initialization",
    "auto_device_selection",
    "freeze_transition",
    "optimizer_preservation",
    "scheduler_preservation",
    "amp_scaler_preservation",
    "epoch_counter_preservation",
    "best_fitness_preservation",
    "resume_behavior",
]
failed = [name for name in required if not result[name]]
if failed or result["training_started"]:
    raise RuntimeError({"failed": failed, "training_started": result["training_started"]})

report = {
    "phase": "10.1 Final Training Launcher Fix",
    "launcher": "train.py",
    "device_policy": "CUDA, then Apple MPS, then CPU",
    "freeze_schedule": {"epochs_1_10": "layers 0-10 frozen", "epochs_11_80": "full model unfrozen"},
    "single_training_run": True,
    "resume_state": ["model/EMA", "optimizer", "scheduler", "AMP scaler", "epoch", "best fitness"],
    "smoke_test": result,
    "training_readiness": "PROJECT READY FOR FINAL TRAINING",
    "training_started": False,
}
path = write_json(ROOT / "reports" / f"phase_10_1_final_launcher_readiness_{utc_timestamp()}.json", report)
print({"report": str(path), "status": report["training_readiness"], "training_started": False})